# Metronome Compass Calibration

A Python port of RNGReporter's HGSS **Seed to Time** verification panel, built for
gathering Metronome-compass calibration data.

Give it a target datetime + delay and a search window, and it enumerates every
candidate seed nearby.  For each seed it reports:

- **Roamer relocation** — where Raikou / Entei / Latios(Latias) move to when the save
  is reloaded (given where they are now).
- **Elm phone-call sequence** — the `P`/`E`/`K` calls you can read off in-game to
  confirm exactly which seed you landed on.

Both come from walking the same LCRNG stream that `times.calculate_seed` seeds.


In [16]:
%load_ext autoreload
%autoreload 2
import datetime as dt
from claytonlib.safari import advance_rng


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## The math

The 32-bit initial seed is exactly what `calculate_seed(time, delay)` produces.  From
it we walk the LCRNG

$$\text{advance\_rng}(s) = (s \cdot \texttt{0x41C64E6D} + \texttt{0x6073}) \bmod 2^{32}$$

taking the **high 16 bits** of each advance as one "call".  The stream is consumed in a
fixed order:

1. **Roamers relocate** — Raikou, then Entei, then Latios/Latias.  Each re-rolls until it
   lands on a route *different* from its current one, so we pass in each roamer's current
   route and get back where it moves to (plus the number of calls consumed).
   - Raikou / Entei use `RouteFromRngJ`; Latios/Latias uses `RouteFromRngK`.
2. **Elm's phone calls** continue on the *same* stream, right after the roamer rolls:
   `call % 3` → `0 = E`, `1 = K`, `2 = P`.

Because only *roaming* legendaries consume roamer rolls, the Elm sequence's starting
offset depends on how many roamers are still free.

> **Seed note:** the initial seed's low 16 bits are `(delay + year - 2000) & 0xFFFF`, so a 2025 date adds 25 vs. a year-2000 date.  `times.calculate_seed` assumes year 2000 and omits this, so this notebook uses its own `seed_for` to match RNGReporter on real dates.


In [17]:
def seed_for(time: dt.datetime, delay: int) -> int:
    """Gen-4 HGSS initial seed for a *real* datetime + delay.

    times.calculate_seed assumes year 2000 (all its dates are 2000-..), so it drops
    the year term.  RNGReporter folds (year - 2000) into the low 16 bits, so we do too
    (Pandora.cs: "[CCCC] includes Year/Delay", CCCC = (Year - 2000) + Delay):
      AA   = (month*day + minute + second) & 0xFF   -> bits 24..31
      BB   = hour                                    -> bits 16..23
      CCCC = (delay + year - 2000) & 0xFFFF          -> bits 0..15
    """
    aa = (time.month * time.day + time.minute + time.second) & 0xFF
    cccc = (delay + time.year - 2000) & 0xFFFF
    return (aa << 24) | (time.hour << 16) | cccc


def route_from_rng_j(rng16: int) -> int:
    """Raikou / Entei route from a 16-bit RNG value (RNGReporter RouteFromRngJ)."""
    m = rng16 & 15
    return m + 29 if m < 11 else m + 31   # routes {29..39, 42..46}


def route_from_rng_k(rng16: int) -> int:
    """Latios / Latias route from a 16-bit RNG value (RNGReporter RouteFromRngK)."""
    m = rng16 % 25
    if m < 22:
        return m + 1                      # routes 1..22
    return {22: 24, 23: 26, 24: 28}[m]    # routes {24, 26, 28}


# (present, previous-route, mapper, output-key) in the fixed relocation order.
_ROAMER_ORDER = ("r", "e", "l")
_ROAMER_MAPPERS = {"r": route_from_rng_j, "e": route_from_rng_j, "l": route_from_rng_k}


def roamer_positions(seed, prev_routes, present):
    """Walk the roamer relocation rolls from `seed`.

    prev_routes: dict like {"r": 31, "e": 30, "l": 0} of each roamer's CURRENT route
                 (0 / anything not on its route table forces the first roll to stick).
    present:     dict like {"r": True, "e": True, "l": False} of which roamers still roam.

    Returns (routes, rng_calls, state) where routes maps r/e/l -> new route (or None if
    that roamer isn't roaming) and state is the LCRNG state after the roamer rolls, ready
    for the Elm calls.
    """
    state = seed
    calls = 0
    routes = {"r": None, "e": None, "l": None}
    for key in _ROAMER_ORDER:
        if not present.get(key):
            continue
        mapper = _ROAMER_MAPPERS[key]
        prev = prev_routes.get(key, 0)
        while True:
            state = advance_rng(state)
            calls += 1
            route = mapper(state >> 16)
            if route != prev:
                routes[key] = route
                break
    return routes, calls, state


def elm_calls(state, count):
    """`count` Elm responses continuing from LCRNG `state`.  0=E, 1=K, 2=P."""
    out = []
    for _ in range(count):
        state = advance_rng(state)
        out.append("EKP"[(state >> 16) % 3])
    return out


def generate_roamer_candidates_near(
    target_time: dt.datetime,
    target_delay: int,
    seconds_window: int,
    delay_window: int,
    prev_routes: dict,
    present: dict,
    elm_count: int = 15,
    match_parity: bool = False,
):
    """Every seed within +/-seconds_window seconds and +/-delay_window delays of
    (target_time, target_delay), annotated with its roamer relocation and Elm calls.

    Returns a list of dicts (one per unique seed) with keys:
      seed, time, delay, sec_delta, delay_delta,
      r_route, e_route, l_route, rng_calls, elm (str), elm_list
    Rows are grouped by time (chronological) then delay ascending.
    match_parity=True keeps only delays with the same even/odd parity as
    target_delay (a real hardware hit lands on one parity).
    """
    by_seed: dict[int, tuple] = {}
    for sec in range(-seconds_window, seconds_window + 1):
        t = target_time + dt.timedelta(seconds=sec)
        for delay in range(target_delay - delay_window, target_delay + delay_window + 1):
            if delay < 0:
                continue
            if match_parity and (delay % 2) != (target_delay % 2):
                continue
            seed = seed_for(t, delay)
            key = (abs(delay - target_delay), abs(sec), seed)
            if seed in by_seed and by_seed[seed][0] <= key:
                continue
            routes, calls, state = roamer_positions(seed, prev_routes, present)
            elm = elm_calls(state, elm_count)
            by_seed[seed] = (key, {
                "seed": seed,
                "time": t,
                "delay": delay,
                "sec_delta": sec,
                "delay_delta": delay - target_delay,
                "r_route": routes["r"],
                "e_route": routes["e"],
                "l_route": routes["l"],
                "rng_calls": calls,
                "elm": "".join(elm),
                "elm_list": elm,
            })
    rows = [entry for _, entry in by_seed.values()]
    rows.sort(key=lambda c: (c["time"], c["delay"]))
    return rows


def print_roamer_candidates(rows, limit=20):
    print(f"{len(rows)} candidate seed(s)\n")
    if limit is None:
        limit = len(rows)
    print(f"  {'Seed':>10}  {'Time':>19}  {'Delay':>6}  {'dD':>4}  {'ds':>3}  "
          f"{'R':>3} {'E':>3} {'L':>3}  {'#':>2}  Elm")
    for c in rows[:limit]:
        fmt = lambda v: f"{v:>3}" if v is not None else "  ."
        print(f"  0x{c['seed']:08X}  {c['time'].strftime('%Y-%m-%d %H:%M:%S')}  "
              f"{c['delay']:>6}  {c['delay_delta']:>+4}  {c['sec_delta']:>+3}  "
              f"{fmt(c['r_route'])} {fmt(c['e_route'])} {fmt(c['l_route'])}  "
              f"{c['rng_calls']:>2}  {c['elm']}")
    if len(rows) > limit:
        print(f"  ... and {len(rows) - limit} more")


## Configure your target

Set your target datetime/delay, the search window, and each roamer's **current** route
(where it is *now*, before the reset) plus whether it's still roaming.  Use `0` for a
current route you don't know or care about.  Results are a plain list of dicts in
`roamer_candidates`, sorted so the exact target sits on top.


In [20]:
# --- Configure your target + current roamer state here ---
target_time    = dt.datetime(2025, 7, 24, 14, 45, 55)
target_delay   = 681
seconds_window = 1        # +/- X seconds
delay_window   = 60       # +/- Y delays

# Only show delays with the same parity (even/odd) as target_delay -- a real
# hardware hit lands on one parity, so this halves the noise.
match_parity   = True

# How many rows to print (None = all).
display_limit  = 40

# Each roamer's CURRENT route (where it is now, before the reset) and whether it is
# still roaming.  Use 0 for the current route if you don't know / don't care.
prev_routes = {"r": 39, "e": 39, "l": 14}
present     = {"r": True, "e": True, "l": True}

roamer_candidates = generate_roamer_candidates_near(
    target_time, target_delay, seconds_window, delay_window,
    prev_routes=prev_routes, present=present, match_parity=match_parity,
)
print_roamer_candidates(roamer_candidates, limit=display_limit)


183 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E0286  2025-07-24 14:45:54     621   -60   -1   29  45  20   3  EEKEEKKKPPPKPKP
  0x0B0E0288  2025-07-24 14:45:54     623   -58   -1   43  32  21   3  KPPKKEEEKPKPKEK
  0x0B0E028A  2025-07-24 14:45:54     625   -56   -1   38  37  24   3  PEPEEPKPPPKEEKE
  0x0B0E028C  2025-07-24 14:45:54     627   -54   -1   35  44  26   3  EPPKEEKPKPEEEEE
  0x0B0E028E  2025-07-24 14:45:54     629   -52   -1   31  31   1   3  PEPEKPEKPPPPPPP
  0x0B0E0290  2025-07-24 14:45:54     631   -50   -1   46  37   2   3  EEPKEKPEKPKEKKK
  0x0B0E0292  2025-07-24 14:45:54     633   -48   -1   42  44   4   3  KKPPKKEPEKKEEPE
  0x0B0E0294  2025-07-24 14:45:54     635   -46   -1   37  31   5   3  PEPKPEPPKKEKEKP
  0x0B0E0296  2025-07-24 14:45:54     637   -44   -1   34  36   7   3  EPPEEPKKEEEPPKK
  0x0B0E0298  2025-07-24 14:45:54     639   -42   -1   30  43   8   3  KEPEKKEEEEPPKPE
  0x0B0E029A  2025-07-24 14:45:5

## Filter by observed roamer positions

Once you've loaded the save and read the roamer map, enter the observed routes here.
Give one number per **roaming** legendary in **R E L** order, space-separated
(e.g. `38 42 11`).  Use `.` for a roamer you can't read or don't want to pin down.
Only the roamers marked `present` in the config above are expected.


In [21]:
def filter_by_observed_rel(candidates, present, observed=None, limit=None):
    """Keep only candidates whose roamer routes match an observed readout.

    Enter one route per roaming legendary in R E L order, space-separated
    (e.g. "38 42 11").  "." leaves that roamer unconstrained.  Pass observed=
    explicitly to skip the prompt.
    """
    keys = [k for k in ("r", "e", "l") if present.get(k)]
    if observed is None:
        labels = " ".join(k.upper() for k in keys)
        observed = input(f"Observed roamer routes ({labels}, space-separated, . = any): ")
    tokens = observed.split()
    if len(tokens) != len(keys):
        raise ValueError(f"expected {len(keys)} value(s) "
                         f"({' '.join(k.upper() for k in keys)}), got {len(tokens)}: {observed!r}")
    wanted = {k: int(tok) for k, tok in zip(keys, tokens) if tok not in (".", "-", "*", "?")}
    matched = [c for c in candidates
               if all(c[f"{k}_route"] == v for k, v in wanted.items())]
    shown = " ".join(f"{k.upper()}={t}" for k, t in zip(keys, tokens))
    print(f"\nObserved {shown}  ->  {len(matched)} / {len(candidates)} candidate(s) match\n")
    print_roamer_candidates(matched, limit=display_limit if limit is None else limit)
    return matched


observed_candidates = filter_by_observed_rel(roamer_candidates, present)


Observed roamer routes (R E L, space-separated, . = any):  35 44 26



Observed R=35 E=44 L=26  ->  3 / 183 candidate(s) match

3 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E028C  2025-07-24 14:45:54     627   -54   -1   35  44  26   3  EPPKEEKPKPEEEEE
  0x0C0E028C  2025-07-24 14:45:55     627   -54   +0   35  44  26   3  PEKEEKPPPEEEKPE
  0x0D0E028C  2025-07-24 14:45:56     627   -54   +1   35  44  26   3  KKEPEPEEEKKPEEE
